# SIREN-Based E-Field Recovery from Spatial Distortion Maps

## Motivation

In a LArTPC, space charge effects (SCE) distort the uniform drift field, causing
ionisation electrons to follow curved trajectories. The observable quantities are
the *spatial distortions* $\boldsymbol{\Delta}(\mathbf{r})$ — the displacement between
an electron's true birth position and its reconstructed position.

This notebook demonstrates that a compact SIREN (Sinusoidal Representation Network)
can learn the distortion field from line-sampled training data, and that the
**electric field can be recovered from the SIREN's analytic derivatives** via
`jax.jvp`, without ever discretising or finite-differencing the field.

### Key result

The E-field recovery requires the *exact* nonlinear Walkowiak drift velocity
inversion $E_x = v^{-1}\!\bigl(v_0 / (1 + \partial\Delta_x/\partial x)\bigr)$.
A naive first-order formula $E_x \approx E_0(1 - \partial\Delta_x/\partial x)$
introduces ~0.9 V/cm systematic error because at $E_0 = 500$ V/cm the drift
velocity is deep in the saturating regime ($v_0/\mu_0 = 1069$ V/cm $\gg E_0$).

### Pipeline

```
SCE Poisson solve  -->  ray-trace distortions  -->  line sampling
    -->  SIREN fit (MSE on corrections)
    -->  JVP for dDelta/dx  -->  nonlinear v(E) inversion  -->  E-field
```

### Required files

| File | Description |
|------|-------------|
| `sce_maps_jaxtpc_41.npz` | Pre-computed SCE maps (41^3 distortion grid, 101^3 E-field grid) |
| `ElectricDistortion/` | SCE simulation package (Poisson solver, ray tracing, Walkowiak v(E)) |

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import jax
import jax.numpy as jnp
import jax.random as jrandom
import equinox as eqx
import optax
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from scipy.interpolate import RegularGridInterpolator

from ElectricDistortion.io.map_io import load_maps_npz
from ElectricDistortion.core.drift_velocity import drift_velocity

plt.rcParams.update({
    'font.size': 11, 'axes.titlesize': 12, 'axes.labelsize': 11,
    'figure.dpi': 150, 'image.cmap': 'RdBu_r',
})

print(f'JAX devices: {jax.devices()}')

## 1. Load SCE Maps and Characterise the Problem

In [ ]:
maps = load_maps_npz('../sce_maps_jaxtpc_41.npz')
params = maps['params']

Lx = params['Lx']   # drift distance (cm)
Ly = params['Ly']   # transverse y (cm)
Lz = params['Lz']   # transverse z (cm)
E0 = params['E0']   # nominal field (V/cm)
T_lar = params.get('temperature', 89.0)

x_out, y_out, z_out = maps['x_grid'], maps['y_grid'], maps['z_grid']
delta_x_true, delta_y_true, delta_z_true = maps['delta_x'], maps['delta_y'], maps['delta_z']
Ex_true, Ey_true, Ez_true = maps['Ex'], maps['Ey'], maps['Ez']
E_ratio = maps['E_ratio']

Nx_p = Ex_true.shape[0]
x_poisson = np.linspace(0, Lx, Nx_p)
y_poisson = np.linspace(0, Ly, Nx_p)
z_poisson = np.linspace(0, Lz, Nx_p)

print(f'Detector: JAXTPC  ({Lx:.0f} x {Ly:.0f} x {Lz:.0f} cm)')
print(f'E0 = {E0:.0f} V/cm,  T = {T_lar} K')
print(f'Distortion grid: {delta_x_true.shape},  E-field grid: {Ex_true.shape}')
print(f'\nMax distortions:  dx={abs(delta_x_true).max():.2f} cm, '
      f'dy={abs(delta_y_true).max():.2f} cm, dz={abs(delta_z_true).max():.2f} cm')
print(f'E-field ratio |E|/E0:  [{E_ratio.min():.4f}, {E_ratio.max():.4f}]  '
      f'=> epsilon ~ {max(abs(1-E_ratio.min()), abs(E_ratio.max()-1)):.1%}')

# Characterise the drift velocity nonlinearity at the operating point
v0 = drift_velocity(E0, T=T_lar)
dE = 0.01
mu0 = (drift_velocity(E0 + dE, T=T_lar) - drift_velocity(E0 - dE, T=T_lar)) / (2 * dE)
E_eff = v0 / mu0

print(f'\nWalkowiak drift velocity at E0:')
print(f'  v0 = {v0:.6f} cm/us')
print(f'  mu0 = dv/dE|_E0 = {mu0:.2e} cm/us per V/cm')
print(f'  v0/mu0 = {E_eff:.1f} V/cm  (ratio to E0: {E_eff/E0:.2f}x)')
print(f'  => v(E) is {"strongly" if E_eff/E0 > 1.5 else "mildly"} nonlinear at this operating point')

## 2. Training Data: Line Sampling

We sample the ground-truth distortion field along random surface-to-surface lines
through the detector volume. This mimics the kind of data available from cosmic
ray tracks (straight-line trajectories with known endpoints). Each line is sampled
at `N_PER_LINE` evenly spaced points, and the ground-truth correction
$\boldsymbol{\Delta}(x,y,z)$ is queried via trilinear interpolation.

In [ ]:
corr_interps = [
    RegularGridInterpolator((x_out, y_out, z_out), d,
                            method='linear', bounds_error=False, fill_value=0.0)
    for d in [delta_x_true, delta_y_true, delta_z_true]
]

efield_interps = [
    RegularGridInterpolator((x_poisson, y_poisson, z_poisson), f,
                            method='linear', bounds_error=False, fill_value=fv)
    for f, fv in [(Ex_true, E0), (Ey_true, 0.0), (Ez_true, 0.0)]
]


def generate_random_lines(n_lines, seed=0):
    """Random surface-to-surface lines, weighted by face area."""
    rng = np.random.RandomState(seed)
    bmin, bmax = np.zeros(3), np.array([Lx, Ly, Lz])
    sizes = bmax - bmin
    areas = np.array([sizes[1]*sizes[2], sizes[1]*sizes[2],
                      sizes[0]*sizes[2], sizes[0]*sizes[2],
                      sizes[0]*sizes[1], sizes[0]*sizes[1]])
    probs = areas / areas.sum()

    def _surface_point():
        face = rng.choice(6, p=probs)
        u, v = rng.uniform(), rng.uniform()
        coords = {
            0: [bmin[0], bmin[1]+u*sizes[1], bmin[2]+v*sizes[2]],
            1: [bmax[0], bmin[1]+u*sizes[1], bmin[2]+v*sizes[2]],
            2: [bmin[0]+u*sizes[0], bmin[1], bmin[2]+v*sizes[2]],
            3: [bmin[0]+u*sizes[0], bmax[1], bmin[2]+v*sizes[2]],
            4: [bmin[0]+u*sizes[0], bmin[1]+v*sizes[1], bmin[2]],
            5: [bmin[0]+u*sizes[0], bmin[1]+v*sizes[1], bmax[2]],
        }
        return np.array(coords[face])

    starts = np.array([_surface_point() for _ in range(n_lines)])
    ends = np.array([_surface_point() for _ in range(n_lines)])
    return starts, ends


def build_line_dataset(n_lines, n_per_line=20, seed=0):
    """Sample points along random lines and query correction ground truth."""
    starts, ends = generate_random_lines(n_lines, seed=seed)
    t = np.linspace(0, 1, n_per_line)[None, :, None]
    positions = (starts[:, None, :] * (1 - t) + ends[:, None, :] * t).reshape(-1, 3)
    corrections = np.stack([interp(positions) for interp in corr_interps], axis=-1)
    return positions.astype(np.float32), corrections.astype(np.float32)

## 3. SIREN Model

A SIREN ([Sitzmann et al., NeurIPS 2020](https://arxiv.org/abs/2006.09661)) uses
$\sin(\omega_0 \cdot (Wx + b))$ activations, which produce smooth, infinitely
differentiable outputs whose derivatives can be computed exactly via `jax.jvp`.
This is essential for E-field recovery, which requires
$\partial\boldsymbol{\Delta}/\partial x$.

**Architecture:** 3 hidden layers of 128 units, $\omega_0 = 5$, ~34k parameters.

**Boundary condition:** The output is multiplied by $(u + 1)$ where
$u = (x - L_x/2)/(L_x/2) \in [-1, 1]$, enforcing $\boldsymbol{\Delta} = 0$ at the
anode ($x = 0 \Rightarrow u = -1$). This is a hard constraint that the network
cannot violate.

In [ ]:
norm_offsets = np.array([Lx / 2, Ly / 2, Lz / 2], dtype=np.float32)
norm_scales = np.array([Lx / 2, Ly / 2, Lz / 2], dtype=np.float32)
norm_offsets_jax = jnp.array(norm_offsets)
norm_scales_jax = jnp.array(norm_scales)


class SirenLayer(eqx.Module):
    weight: jnp.ndarray
    bias: jnp.ndarray
    omega_0: float = eqx.field(static=True)

    def __init__(self, in_f, out_f, omega_0=5.0, first=False, *, key):
        w_key, b_key = jrandom.split(key)
        bound = 1.0 / in_f if first else jnp.sqrt(6.0 / in_f) / omega_0
        self.weight = jrandom.uniform(w_key, (out_f, in_f), minval=-bound, maxval=bound)
        self.bias = jrandom.uniform(b_key, (out_f,), minval=-bound, maxval=bound)
        self.omega_0 = omega_0

    def __call__(self, x):
        return jnp.sin(self.omega_0 * (self.weight @ x + self.bias))


class Siren3D(eqx.Module):
    layers: list
    output_layer: eqx.nn.Linear

    def __init__(self, hidden_features=128, hidden_layers=3, omega_0=5.0, *, key):
        keys = jrandom.split(key, hidden_layers + 2)
        self.layers = [SirenLayer(3, hidden_features, omega_0=omega_0, first=True, key=keys[0])]
        for i in range(1, hidden_layers):
            self.layers.append(SirenLayer(hidden_features, hidden_features,
                                         omega_0=omega_0, first=False, key=keys[i]))
        w_key, b_key = jrandom.split(keys[hidden_layers])
        bound = jnp.sqrt(6.0 / hidden_features) / omega_0
        self.output_layer = eqx.nn.Linear(hidden_features, 3, key=keys[hidden_layers + 1])
        self.output_layer = eqx.tree_at(
            lambda l: (l.weight, l.bias), self.output_layer,
            (jrandom.uniform(w_key, (3, hidden_features), minval=-bound, maxval=bound),
             jrandom.uniform(b_key, (3,), minval=-bound, maxval=bound)))

    def __call__(self, coords_norm):
        x = coords_norm
        for layer in self.layers:
            x = layer(x)
        raw = self.output_layer(x)
        return raw * (coords_norm[0] + 1.0)


n_params = sum(x.size for x in jax.tree.leaves(eqx.filter(Siren3D(key=jrandom.PRNGKey(0)), eqx.is_array)))
print(f'SIREN parameters: {n_params:,}')

## 4. Training

Standard MSE loss on the correction values $\|\hat{\boldsymbol{\Delta}} - \boldsymbol{\Delta}\|^2$.
The training data is batched by line (256 lines/batch = 25,600 points) with a
warmup + cosine-decay learning rate schedule.

In [ ]:
from tqdm.auto import tqdm

N_LINES = 1000
N_PER_LINE = 100
N_EPOCHS = 2000
LINES_PER_BATCH = 256

train_pos, train_corr = build_line_dataset(N_LINES, N_PER_LINE, seed=42)
pos_norm = jnp.array((train_pos - norm_offsets) / norm_scales)
corr_jax = jnp.array(train_corr)

pos_by_line = pos_norm.reshape(N_LINES, N_PER_LINE, 3)
corr_by_line = corr_jax.reshape(N_LINES, N_PER_LINE, 3)
n_batches = max(1, N_LINES // LINES_PER_BATCH)

print(f'Training: {N_LINES} lines x {N_PER_LINE} pts = {N_LINES * N_PER_LINE:,} samples')
print(f'Batching: {LINES_PER_BATCH} lines/batch, {n_batches} batches/epoch')
print(f'Correction range: [{train_corr.min():.3f}, {train_corr.max():.3f}] cm')

key = jrandom.PRNGKey(0)
model = Siren3D(key=key)

total_steps = N_EPOCHS * n_batches
schedule = optax.warmup_cosine_decay_schedule(
    init_value=1e-3 * 0.01, peak_value=1e-3,
    warmup_steps=50 * n_batches, decay_steps=total_steps,
    end_value=1e-3 * 0.01)
optimizer = optax.adam(schedule)
opt_state = optimizer.init(eqx.filter(model, eqx.is_array))


@eqx.filter_jit
def train_step(model, opt_state, batch_pos, batch_corr):
    def loss_fn(model):
        pred = jax.vmap(model)(batch_pos)
        return jnp.mean((pred - batch_corr) ** 2)
    loss, grads = eqx.filter_value_and_grad(loss_fn)(model)
    updates, new_opt_state = optimizer.update(grads, opt_state, model)
    return eqx.apply_updates(model, updates), new_opt_state, loss


losses = []
t0 = time.time()
rng = np.random.RandomState(0)

pbar = tqdm(range(N_EPOCHS), desc='Training')
for epoch in pbar:
    line_perm = rng.permutation(N_LINES)
    epoch_loss = 0.0
    for b in range(n_batches):
        line_idx = line_perm[b * LINES_PER_BATCH : (b + 1) * LINES_PER_BATCH]
        batch_pos = pos_by_line[line_idx].reshape(-1, 3)
        batch_corr = corr_by_line[line_idx].reshape(-1, 3)
        model, opt_state, loss = train_step(model, opt_state, batch_pos, batch_corr)
        epoch_loss += float(loss)
    epoch_loss /= n_batches
    if epoch % 100 == 0:
        losses.append((epoch, epoch_loss))
    pbar.set_postfix(loss=f'{epoch_loss:.6f}')

losses.append((N_EPOCHS, epoch_loss))
elapsed = time.time() - t0
print(f'Final loss: {epoch_loss:.6f}  |  Training time: {elapsed:.1f}s')

## 5. Evaluate Spatial Corrections

In [ ]:
GX, GY, GZ = np.meshgrid(x_out, y_out, z_out, indexing='ij')
eval_pos = np.stack([GX.ravel(), GY.ravel(), GZ.ravel()], axis=-1).astype(np.float32)
eval_norm = jnp.array((eval_pos - norm_offsets) / norm_scales)

pred_corr = np.array(jax.vmap(model)(eval_norm))
pred_dx = pred_corr[:, 0].reshape(GX.shape)
pred_dy = pred_corr[:, 1].reshape(GX.shape)
pred_dz = pred_corr[:, 2].reshape(GX.shape)

err_dx = pred_dx - delta_x_true
err_dy = pred_dy - delta_y_true
err_dz = pred_dz - delta_z_true

print('Correction MAE (cm):')
print(f'  dx: {np.mean(np.abs(err_dx)):.4f}  (max true: {np.abs(delta_x_true).max():.3f})')
print(f'  dy: {np.mean(np.abs(err_dy)):.4f}  (max true: {np.abs(delta_y_true).max():.3f})')
print(f'  dz: {np.mean(np.abs(err_dz)):.4f}  (max true: {np.abs(delta_z_true).max():.3f})')

## 6. E-Field Recovery via Exact Drift Velocity Inversion

### Derivation

The distortion $\Delta_x$ is defined as $\Delta_x = v_0 \cdot t_\mathrm{drift} - x_0$,
where $v_0 = v(E_0)$ is the nominal drift velocity. In the 1D approximation
(valid when $E_y, E_z \ll E_x$):

$$t_\mathrm{drift}(x_0) = \int_0^{x_0} \frac{dx}{v\bigl(E_x(x)\bigr)}$$

Differentiating $x_0 + \Delta_x = v_0 \cdot t_\mathrm{drift}$ with respect to $x_0$:

$$1 + \frac{\partial \Delta_x}{\partial x} = \frac{v_0}{v\bigl(E_x(x_0)\bigr)}
\quad\Longrightarrow\quad
v\bigl(E_x\bigr) = \frac{v_0}{1 + \partial\Delta_x/\partial x}$$

The electric field is then recovered by **inverting the drift velocity function**:
$$\boxed{E_x = v^{-1}\!\left(\frac{v_0}{1 + \partial\Delta_x/\partial x}\right)}$$

### Why the first-order formula fails

The naive formula $E_x \approx E_0(1 - \partial\Delta_x/\partial x)$ assumes $v(E) \propto E$
(constant mobility). But the Walkowiak parameterisation at 500 V/cm gives
$v_0/(\mu_0 \cdot E_0) = 2.14$, meaning the drift velocity is 2x what a linear
extrapolation from the local slope would predict. This causes the naive formula to
underestimate E-field variations by roughly a factor of 2.

### Transverse components

The transverse deflection satisfies $dy/dx = E_y/E_x$, so
$\partial\Delta_y/\partial x \approx E_y/E_0$. The $v(E)$ nonlinearity cancels in the ratio,
and the first-order formula $E_y = -E_0 \cdot \partial\Delta_y/\partial x$ is correct.

In [ ]:
# Lookup table for v_inv: given v_target, return E
_E_table = np.linspace(1.0, 2000.0, 20000)
_v_table = np.array([drift_velocity(e, T=T_lar) for e in _E_table])
_v_inv_interp = RegularGridInterpolator(
    (_v_table,), _E_table, method='linear',
    bounds_error=False, fill_value=E0)


def make_model_physical(model):
    """Wrap model to accept physical coordinates (cm)."""
    def model_physical(xyz_phys):
        return model((xyz_phys - norm_offsets_jax) / norm_scales_jax)
    return model_physical


@jax.jit
def compute_dDelta_dx_batch(model, positions):
    """dDelta/dx via forward-mode AD (JVP with tangent along x)."""
    model_phys = make_model_physical(model)
    tangent = jnp.array([1.0, 0.0, 0.0])
    return jax.vmap(lambda xyz: jax.jvp(model_phys, (xyz,), (tangent,))[1])(positions)


# Compute SIREN dDelta/dx on evaluation grid
eval_pos_jax = jnp.array(eval_pos)
dDelta_dx_all = np.array(compute_dDelta_dx_batch(model, eval_pos_jax))
dDdx_x = dDelta_dx_all[:, 0].reshape(GX.shape)
dDdx_y = dDelta_dx_all[:, 1].reshape(GX.shape)
dDdx_z = dDelta_dx_all[:, 2].reshape(GX.shape)

# E_x: exact nonlinear inversion
v_target = v0 / (1.0 + dDdx_x)
pred_Ex = _v_inv_interp(v_target.ravel()).reshape(GX.shape)

# E_y, E_z: first-order (exact for transverse components)
pred_Ey = -E0 * dDdx_y
pred_Ez = -E0 * dDdx_z

# Comparison: old (wrong) first-order formula
pred_Ex_old = E0 * (1.0 - dDdx_x)

# Ground truth
true_Ex_eval = efield_interps[0](eval_pos).reshape(GX.shape)
true_Ey_eval = efield_interps[1](eval_pos).reshape(GX.shape)
true_Ez_eval = efield_interps[2](eval_pos).reshape(GX.shape)
true_Emag_eval = np.sqrt(true_Ex_eval**2 + true_Ey_eval**2 + true_Ez_eval**2)
pred_Emag = np.sqrt(pred_Ex**2 + pred_Ey**2 + pred_Ez**2)
Emag_rel_err = np.abs(pred_Emag - true_Emag_eval) / true_Emag_eval

print('OLD formula  E_x = E0*(1 - dDx/dx):')
print(f'  Ex MAE = {np.mean(np.abs(pred_Ex_old - true_Ex_eval)):.3f} V/cm\n')

print('NEW formula  E_x = v_inv(v0 / (1 + dDx/dx)):')
print(f'  Ex MAE = {np.mean(np.abs(pred_Ex - true_Ex_eval)):.3f} V/cm  '
      f'(range: [{true_Ex_eval.min():.1f}, {true_Ex_eval.max():.1f}])')
print(f'  Ey MAE = {np.mean(np.abs(pred_Ey - true_Ey_eval)):.3f} V/cm  '
      f'(range: [{true_Ey_eval.min():.1f}, {true_Ey_eval.max():.1f}])')
print(f'  Ez MAE = {np.mean(np.abs(pred_Ez - true_Ez_eval)):.3f} V/cm  '
      f'(range: [{true_Ez_eval.min():.1f}, {true_Ez_eval.max():.1f}])')
print(f'\n  |E| rel error:  mean = {Emag_rel_err.mean():.4%},  max = {Emag_rel_err.max():.4%}')


def box_recomb(E_Vcm, dedx=2.0):
    xi = (0.212 / 1.396) * dedx / np.maximum(E_Vcm / 1000.0, 1e-10)
    return np.log(0.93 + xi) / xi

R_true = box_recomb(true_Emag_eval)
R_pred = box_recomb(pred_Emag)
R_rel_err = np.abs(R_pred - R_true) / R_true
print(f'  R rel error:    mean = {R_rel_err.mean():.4%},  max = {R_rel_err.max():.4%}')

## 7. Visualisation

### Plotting utilities

In [ ]:
def plot_correction_slices(true_3d, pred_3d, label,
                           x_grid, y_grid, z_grid, save_path=None):
    """YZ slices at 25%, 50%, 75% drift: truth, SIREN, difference."""
    err = pred_3d - true_3d
    nx = true_3d.shape[0]
    x_indices = [nx // 4, nx // 2, 3 * nx // 4]

    fig, axes = plt.subplots(3, 3, figsize=(14, 11))
    for row, ix in enumerate(x_indices):
        s_true, s_pred, s_err = true_3d[ix], pred_3d[ix], err[ix]
        vmax_val = max(np.abs(s_true).max(), np.abs(s_pred).max(), 1e-6)
        norm_val = TwoSlopeNorm(vmin=-vmax_val, vcenter=0, vmax=vmax_val)
        err_max = max(np.abs(s_err).max(), 0.01)
        norm_err = TwoSlopeNorm(vmin=-err_max, vcenter=0, vmax=err_max)

        for col, (data, norm, ttl) in enumerate([
            (s_true, norm_val, 'Truth'), (s_pred, norm_val, 'SIREN'), (s_err, norm_err, 'Difference')]):
            ax = axes[row, col]
            im = ax.pcolormesh(y_grid, z_grid, data.T, norm=norm, cmap='RdBu_r', shading='auto')
            cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            if col == 2: cb.set_label('cm')
            ax.set_xlabel('y (cm)')
            ax.set_ylabel(f'x = {x_grid[ix]:.0f} cm\n\nz (cm)' if col == 0 else 'z (cm)')
            if row == 0: ax.set_title(ttl, fontweight='bold')

    fig.suptitle(f'{label} Correction — YZ slices (cm)', fontsize=14, fontweight='bold', y=0.98)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    if save_path: fig.savefig(save_path, bbox_inches='tight')
    plt.show()


def plot_efield_slices(true_3d, pred_3d, label, unit,
                       x_grid, y_grid, z_grid, save_path=None):
    """YZ slices at 25%, 50%, 75% drift: truth, SIREN, difference."""
    err = pred_3d - true_3d
    nx = true_3d.shape[0]
    x_indices = [nx // 4, nx // 2, 3 * nx // 4]

    fig, axes = plt.subplots(3, 3, figsize=(14, 11))
    for row, ix in enumerate(x_indices):
        s_true, s_pred, s_err = true_3d[ix], pred_3d[ix], err[ix]
        vmin_val = min(s_true.min(), s_pred.min())
        vmax_val = max(s_true.max(), s_pred.max())
        err_max = max(np.abs(s_err).max(), 0.01)
        norm_err = TwoSlopeNorm(vmin=-err_max, vcenter=0, vmax=err_max)

        for col, (data, ttl) in enumerate([
            (s_true, 'Truth'), (s_pred, 'SIREN (exact)'), (s_err, 'Difference')]):
            ax = axes[row, col]
            if col < 2:
                im = ax.pcolormesh(y_grid, z_grid, data.T, vmin=vmin_val, vmax=vmax_val,
                                   cmap='RdBu_r', shading='auto')
            else:
                im = ax.pcolormesh(y_grid, z_grid, data.T, norm=norm_err,
                                   cmap='RdBu_r', shading='auto')
            cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            if col == 2: cb.set_label(unit)
            ax.set_xlabel('y (cm)')
            ax.set_ylabel(f'x = {x_grid[ix]:.0f} cm\n\nz (cm)' if col == 0 else 'z (cm)')
            if row == 0: ax.set_title(ttl, fontweight='bold')

    fig.suptitle(f'{label} — YZ slices ({unit})', fontsize=14, fontweight='bold', y=0.98)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    if save_path: fig.savefig(save_path, bbox_inches='tight')
    plt.show()

### 7a. Spatial Correction Slices

In [ ]:
plot_correction_slices(delta_x_true, pred_dx, r'$\Delta x$', x_out, y_out, z_out, save_path='corr_slices_dx.png')
plot_correction_slices(delta_y_true, pred_dy, r'$\Delta y$', x_out, y_out, z_out, save_path='corr_slices_dy.png')
plot_correction_slices(delta_z_true, pred_dz, r'$\Delta z$', x_out, y_out, z_out, save_path='corr_slices_dz.png')

### 7b. E-Field Component Slices

In [ ]:
plot_efield_slices(true_Ex_eval, pred_Ex, r'$E_x$', 'V/cm', x_out, y_out, z_out, save_path='efield_slices_Ex.png')
plot_efield_slices(true_Ey_eval, pred_Ey, r'$E_y$', 'V/cm', x_out, y_out, z_out, save_path='efield_slices_Ey.png')
plot_efield_slices(true_Ez_eval, pred_Ez, r'$E_z$', 'V/cm', x_out, y_out, z_out, save_path='efield_slices_Ez.png')

### 7c. E-Field Magnitude and Recombination

In [ ]:
plot_efield_slices(true_Emag_eval, pred_Emag, r'$|\mathbf{E}|$', 'V/cm',
                   x_out, y_out, z_out, save_path='efield_slices_Emag.png')
plot_efield_slices(R_true, R_pred, 'Recombination $R$', '',
                   x_out, y_out, z_out, save_path='recomb_slices.png')

## 8. Error Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for col, (err, lbl) in enumerate([
    (err_dx.ravel(), r'$\Delta x$'), (err_dy.ravel(), r'$\Delta y$'), (err_dz.ravel(), r'$\Delta z$')]):
    ax = axes[0, col]
    ax.hist(err, bins=80, color='steelblue', edgecolor='none', alpha=0.85)
    ax.axvline(0, color='k', lw=0.8, ls='--')
    ax.set_xlabel(f'{lbl} error (cm)'); ax.set_ylabel('Counts')
    ax.set_title(f'{lbl}  MAE={np.mean(np.abs(err)):.4f} cm', fontweight='bold')

for col, (err, lbl, unit) in enumerate([
    ((pred_Ex - true_Ex_eval).ravel(), r'$E_x$', 'V/cm'),
    (((pred_Emag - true_Emag_eval) / true_Emag_eval * 100).ravel(), r'$|E|$ relative', '%'),
    (((R_pred - R_true) / R_true * 100).ravel(), '$R$ relative', '%')]):
    ax = axes[1, col]
    ax.hist(err, bins=80, color='coral', edgecolor='none', alpha=0.85)
    ax.axvline(0, color='k', lw=0.8, ls='--')
    ax.set_xlabel(f'{lbl} error ({unit})'); ax.set_ylabel('Counts')
    ax.set_title(f'{lbl}  RMS={np.sqrt(np.mean(err**2)):.4f} {unit}', fontweight='bold')

fig.suptitle('Error Distributions', fontsize=14, fontweight='bold')
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig('error_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for col, (err_3d, lbl) in enumerate([
    (np.abs(err_dx), r'$|\Delta x$ error$|$'),
    (np.abs(err_dy), r'$|\Delta y$ error$|$'),
    (np.abs(err_dz), r'$|\Delta z$ error$|$')]):
    ax = axes[col]
    profile_mean = err_3d.mean(axis=(1, 2))
    profile_max = err_3d.max(axis=(1, 2))
    ax.fill_between(x_out, 0, profile_max, alpha=0.25, color='steelblue', label='Max')
    ax.plot(x_out, profile_mean, 'o-', ms=3, color='steelblue', lw=1.5, label='Mean')
    ax.set_xlabel('x (cm)  [anode=0, cathode=216]'); ax.set_ylabel('Correction error (cm)')
    ax.set_title(lbl, fontweight='bold'); ax.legend(frameon=False)
    ax.set_xlim(0, Lx); ax.grid(alpha=0.3)

fig.suptitle('Correction Error vs Drift Position', fontsize=13, fontweight='bold')
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig('error_vs_drift.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

Emag_rel_pct = np.abs(pred_Emag - true_Emag_eval) / true_Emag_eval * 100
R_rel_pct = np.abs(R_pred - R_true) / R_true * 100

for col, (err_3d, lbl, unit) in enumerate([
    (np.abs(pred_Ex - true_Ex_eval), r'$|E_x$ error$|$', 'V/cm'),
    (Emag_rel_pct, r'$|E|$ relative error', '%'),
    (R_rel_pct, '$R$ relative error', '%')]):
    ax = axes[col]
    profile_mean = err_3d.mean(axis=(1, 2))
    profile_max = err_3d.max(axis=(1, 2))
    ax.fill_between(x_out, 0, profile_max, alpha=0.25, color='coral', label='Max')
    ax.plot(x_out, profile_mean, 'o-', ms=3, color='coral', lw=1.5, label='Mean')
    ax.set_xlabel('x (cm)  [anode=0, cathode=216]'); ax.set_ylabel(f'{lbl} ({unit})')
    ax.set_title(lbl, fontweight='bold'); ax.legend(frameon=False)
    ax.set_xlim(0, Lx); ax.grid(alpha=0.3)

fig.suptitle('E-Field & Recombination Error vs Drift Position', fontsize=13, fontweight='bold')
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig('efield_error_vs_drift.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
epochs_arr, loss_arr = zip(*losses)
ax.semilogy(epochs_arr, loss_arr, 'o-', ms=3, color='steelblue', lw=1.5)
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title(f'Training Curve ({N_LINES} lines, {N_LINES*N_PER_LINE:,} pts)', fontweight='bold')
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig('training_curve.png', bbox_inches='tight')
plt.show()

## 9. Summary

In [ ]:
print('=' * 70)
print(f'SIREN SCE Recovery Summary  (JAXTPC, {N_LINES} lines)')
print('=' * 70)
print(f'\nDetector:  JAXTPC ({Lx:.0f} x {Ly:.0f} x {Lz:.0f} cm, E0={E0:.0f} V/cm, T={T_lar} K)')
print(f'SCE magnitude: epsilon ~ {max(abs(1-E_ratio.min()), abs(E_ratio.max()-1)):.1%}')
print(f'Max displacements: dx={abs(delta_x_true).max():.2f}, '
      f'dy={abs(delta_y_true).max():.2f}, dz={abs(delta_z_true).max():.2f} cm')
print(f'\nv(E) nonlinearity:  v0/mu0 = {E_eff:.0f} V/cm  (ratio to E0: {E_eff/E0:.2f}x)')
print(f'Training: {N_LINES} lines x {N_PER_LINE} pts = {N_LINES*N_PER_LINE:,} samples, '
      f'{N_EPOCHS} epochs, {elapsed:.1f}s')
print(f'SIREN: {n_params:,} parameters, omega_0=5, 3 hidden layers x 128')
print(f'\n{"Metric":<30s}  {"Mean":>10s}  {"Max":>10s}')
print('-' * 55)
print(f'{"Corr dx MAE (cm)":<30s}  {np.mean(np.abs(err_dx)):>10.4f}  {np.abs(err_dx).max():>10.4f}')
print(f'{"Corr dy MAE (cm)":<30s}  {np.mean(np.abs(err_dy)):>10.4f}  {np.abs(err_dy).max():>10.4f}')
print(f'{"Corr dz MAE (cm)":<30s}  {np.mean(np.abs(err_dz)):>10.4f}  {np.abs(err_dz).max():>10.4f}')
print(f'{"Ex MAE (V/cm) — OLD formula":<30s}  {np.mean(np.abs(pred_Ex_old - true_Ex_eval)):>10.3f}')
print(f'{"Ex MAE (V/cm) — NEW formula":<30s}  {np.mean(np.abs(pred_Ex - true_Ex_eval)):>10.3f}')
print(f'{"Ey MAE (V/cm)":<30s}  {np.mean(np.abs(pred_Ey - true_Ey_eval)):>10.3f}')
print(f'{"Ez MAE (V/cm)":<30s}  {np.mean(np.abs(pred_Ez - true_Ez_eval)):>10.3f}')
print(f'{"|E| relative error":<30s}  {Emag_rel_err.mean():>9.4%}  {Emag_rel_err.max():>9.4%}')
print(f'{"R relative error":<30s}  {R_rel_err.mean():>9.4%}  {R_rel_err.max():>9.4%}')
print('=' * 70)